In [55]:
%pip install kernel_tuner
%pip install pycuda
import kernel_tuner

In [56]:
import numpy as np
import kernel_tuner as kt
import pycuda

kernel_name = "matmul"
kernel_code = """
__global__ void matmul(int n, float* A, float* B, float* C) {
  int j = blockIdx.x * blockDim.x + threadIdx.x;
  int i = blockIdx.y * blockDim.y + threadIdx.y;

  if (i < n && j < n) {
    for (int k = 0; k < n; k++) {
      // C[i,j] += A[i,k] * B[k,j]
      C[i * n + j] += A[i * n + k] * B[k * n + j];
    }
  }
}
"""

n = np.int32(500)
A = np.random.rand(n, n).astype("float32")
B = np.random.rand(n, n).astype("float32")
C = np.zeros((n, n)).astype("float32")

problem_size = (n, n)
arguments = [n, A, B, C]
answer = [None, None, None, np.matmul(A, B)]

tune_params = dict()
tune_params["block_size_x"] = [32, 64, 128, 256]

kt.tune_kernel(
    kernel_name,
    kernel_code,
    problem_size,
    arguments,
    tune_params,
    answer=answer,
    lang="nvcuda",
);

Using: Tesla T4
block_size_x=32, time=14.668ms
block_size_x=64, time=11.773ms
block_size_x=128, time=11.967ms
block_size_x=256, time=12.321ms
best performing configuration:
block_size_x=64, time=11.773ms


In [57]:
kernel_name = "matmul"
kernel_code = """
__global__ void matmul(int n, float* A, float* B, float* C) {
  int j = blockIdx.x * blockDim.x + threadIdx.x;
  int i = blockIdx.y * blockDim.y + threadIdx.y;

  if (i < n && j < n) {
    #pragma unroll loop_unroll_factor
    for (int k = 0; k < n; k++) {
      C[i * n + j] += A[i * n + k] * B[k * n + j];
    }
  }
}
"""


tune_params = dict()
tune_params["block_size_x"] = [32, 64, 128, 256]
tune_params["loop_unroll_factor"] = [1, 5, 25]

kt.tune_kernel(
    kernel_name,
    kernel_code,
    problem_size,
    arguments,
    tune_params,
    answer=answer,
    lang="nvcuda",
);

Using: Tesla T4
block_size_x=32, loop_unroll_factor=1, time=11.988ms
block_size_x=32, loop_unroll_factor=5, time=11.667ms
block_size_x=32, loop_unroll_factor=25, time=11.638ms
block_size_x=64, loop_unroll_factor=1, time=12.313ms
block_size_x=64, loop_unroll_factor=5, time=11.892ms
block_size_x=64, loop_unroll_factor=25, time=11.881ms
block_size_x=128, loop_unroll_factor=1, time=12.405ms
block_size_x=128, loop_unroll_factor=5, time=12.016ms
block_size_x=128, loop_unroll_factor=25, time=11.923ms
block_size_x=256, loop_unroll_factor=1, time=12.666ms
block_size_x=256, loop_unroll_factor=5, time=12.299ms
block_size_x=256, loop_unroll_factor=25, time=12.112ms
best performing configuration:
block_size_x=32, loop_unroll_factor=25, time=11.638ms


In [58]:
kernel_name = "matmul"
kernel_code = """
__global__ void matmul(int n, float* A, float* B, float* C) {
  int j = blockIdx.x * blockDim.x + threadIdx.x;
  int i = blockIdx.y * blockDim.y + threadIdx.y;

  if (i < n && j < n) {
    #pragma unroll loop_unroll_factor
    for (int k = 0; k < n; k++) {
      if (use_global_cache) {
        C[i * n + j] += __ldg(&A[i * n + k]) * __ldg(&B[k * n + j]);
      } else {
        C[i * n + j] += A[i * n + k] * B[k * n + j];
      }
    }
  }
}
"""


tune_params = dict()
tune_params["block_size_x"] = [32, 64, 128, 256]
tune_params["loop_unroll_factor"] = [1, 5, 25]
tune_params["use_global_cache"] = [0, 1]

kt.tune_kernel(
    kernel_name,
    kernel_code,
    problem_size,
    arguments,
    tune_params,
    answer=answer,
    lang="nvcuda",
);

Using: Tesla T4
block_size_x=32, loop_unroll_factor=1, use_global_cache=0, time=12.008ms
block_size_x=32, loop_unroll_factor=1, use_global_cache=1, time=12.877ms
block_size_x=32, loop_unroll_factor=5, use_global_cache=0, time=11.555ms
block_size_x=32, loop_unroll_factor=5, use_global_cache=1, time=3.704ms
block_size_x=32, loop_unroll_factor=25, use_global_cache=0, time=11.650ms
block_size_x=32, loop_unroll_factor=25, use_global_cache=1, time=7.264ms
block_size_x=64, loop_unroll_factor=1, use_global_cache=0, time=12.218ms
block_size_x=64, loop_unroll_factor=1, use_global_cache=1, time=13.660ms
block_size_x=64, loop_unroll_factor=5, use_global_cache=0, time=12.007ms
block_size_x=64, loop_unroll_factor=5, use_global_cache=1, time=3.856ms
block_size_x=64, loop_unroll_factor=25, use_global_cache=0, time=11.897ms
block_size_x=64, loop_unroll_factor=25, use_global_cache=1, time=7.415ms
block_size_x=128, loop_unroll_factor=1, use_global_cache=0, time=12.530ms
block_size_x=128, loop_unroll_fact

In [59]:
kernel_name = "matmul"
kernel_code = """
__global__ void matmul(int n, float* A, float* B, float* C) {
  int j = blockIdx.x * blockDim.x + threadIdx.x;
  int i = blockIdx.y * blockDim.y + threadIdx.y;

  if (i < n && j < n) {
    #pragma unroll loop_unroll_factor
    for (int k = 0; k < n; k++) {
      if (use_global_cache) {
        C[i * n + j] += __ldg(&A[i * n + k]) * __ldg(&B[k * n + j]);
      } else {
        C[i * n + j] += A[i * n + k] * B[k * n + j];
      }
    }
  }
}
"""


tune_params = dict()
tune_params["block_size_x"] = [32, 64, 128, 256]
tune_params["block_size_y"] = [1, 4, 16, 32]
tune_params["loop_unroll_factor"] = [1, 5, 25]
tune_params["use_global_cache"] = [0, 1]

restrictions = "block_size_x*block_size_y >= 64 and block_size_x*block_size_y <= 1024"

kt.tune_kernel(
    kernel_name,
    kernel_code,
    problem_size,
    arguments,
    tune_params,
    answer=answer,
    restrictions=restrictions,
    lang="nvcuda",
);

Using: Tesla T4
block_size_x=32, block_size_y=4, loop_unroll_factor=1, use_global_cache=0, time=9.425ms
block_size_x=32, block_size_y=4, loop_unroll_factor=1, use_global_cache=1, time=9.845ms
block_size_x=32, block_size_y=4, loop_unroll_factor=5, use_global_cache=0, time=9.226ms
block_size_x=32, block_size_y=4, loop_unroll_factor=5, use_global_cache=1, time=3.096ms
block_size_x=32, block_size_y=4, loop_unroll_factor=25, use_global_cache=0, time=9.126ms
block_size_x=32, block_size_y=4, loop_unroll_factor=25, use_global_cache=1, time=6.508ms
block_size_x=32, block_size_y=16, loop_unroll_factor=1, use_global_cache=0, time=9.045ms
block_size_x=32, block_size_y=16, loop_unroll_factor=1, use_global_cache=1, time=10.478ms
block_size_x=32, block_size_y=16, loop_unroll_factor=5, use_global_cache=0, time=8.868ms
block_size_x=32, block_size_y=16, loop_unroll_factor=5, use_global_cache=1, time=2.824ms
block_size_x=32, block_size_y=16, loop_unroll_factor=25, use_global_cache=0, time=8.808ms
block_s

In [75]:
kernel_name = "matmul"
kernel_code = """
__global__ void matmul(int n, float* A, float* B, float* C) {
  int j_start = (blockIdx.x * blockDim.x + threadIdx.x) * elements_per_thread;
  int i = blockIdx.y * blockDim.y + threadIdx.y;

  if (i < n && j_start < n) {
    #pragma unroll loop_unroll_factor
    for (int k = 0; k < n; k++) {
      #pragma unroll elements_per_thread
      for (int j_offset = 0; j_offset < elements_per_thread; j_offset++) {
        int j = j_start + j_offset;
        if (use_global_cache) {
          C[i * n + j] += __ldg(&A[i * n + k]) * __ldg(&B[k * n + j]);
        } else {
          C[i * n + j] += A[i * n + k] * B[k * n + j];
        }
      }
    }
  }
}
"""

tune_params = dict()
tune_params["block_size_x"] = [32, 64, 128, 256]
tune_params["block_size_y"] = [1, 4, 16, 32]
tune_params["loop_unroll_factor"] = [1, 5, 25]
tune_params["use_global_cache"] = [0, 1]
tune_params["elements_per_thread"] = [1, 2, 4]

restrictions = "block_size_x*block_size_y >= 64 and block_size_x*block_size_y <= 1024"

kt.tune_kernel(
    kernel_name,
    kernel_code,
    problem_size,
    arguments,
    tune_params,
    answer=answer,
    restrictions=restrictions,
    lang="nvcuda",
);

Using: Tesla T4
block_size_x=32, block_size_y=4, loop_unroll_factor=1, use_global_cache=0, elements_per_thread=1, time=2.184ms
block_size_x=32, block_size_y=4, loop_unroll_factor=1, use_global_cache=0, elements_per_thread=2, time=3.886ms
block_size_x=32, block_size_y=4, loop_unroll_factor=1, use_global_cache=0, elements_per_thread=4, time=7.577ms
block_size_x=32, block_size_y=4, loop_unroll_factor=1, use_global_cache=1, elements_per_thread=1, time=2.147ms
block_size_x=32, block_size_y=4, loop_unroll_factor=1, use_global_cache=1, elements_per_thread=2, time=2.901ms
block_size_x=32, block_size_y=4, loop_unroll_factor=1, use_global_cache=1, elements_per_thread=4, time=5.410ms
block_size_x=32, block_size_y=4, loop_unroll_factor=5, use_global_cache=0, elements_per_thread=1, time=2.174ms
block_size_x=32, block_size_y=4, loop_unroll_factor=5, use_global_cache=0, elements_per_thread=2, time=3.891ms
block_size_x=32, block_size_y=4, loop_unroll_factor=5, use_global_cache=0, elements_per_thread=4